In [1]:
import requests
import os
import sys
import platform
from lakehouse.spark import bronze, silver
from pyspark.sql import DataFrame, SparkSession
from delta import configure_spark_with_delta_pip
from pyspark.sql import functions as F
from pyspark.sql import DataFrame
import json

In [2]:
if platform.system() == "Windows":
    os.environ["PYSPARK_PYTHON"] = sys.executable
    os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable
    print("Adding Python ENV variables on Windows")

Adding Python ENV variables on Windows


In [3]:
builder = (
    SparkSession.builder.appName("Data with Nikk the Greek Spark Session")
    .master("local[4]")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config(
        "spark.sql.catalog.spark_catalog",
        "org.apache.spark.sql.delta.catalog.DeltaCatalog",
    )
)
spark = configure_spark_with_delta_pip(builder).getOrCreate()

In [4]:
CATALOG = spark.catalog.currentCatalog()

# 1. Set Up and Bronze Data

In [5]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.bronze")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.silver")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.gold")

DataFrame[]

In [6]:
options = {"catalog": CATALOG, "target_schema": "bronze"}

In [7]:
@F.udf(returnType="STRING")
def get_properties(url):
    json_request = requests.get(url).json()
    return json.dumps(json_request["result"]["properties"])

In [8]:
class StarWarsBronze(bronze.Bronze):
    def custom_load(self, table):
        results = []
        query = f"https://swapi.tech/api/{table}"
        json_request = requests.get(query).json()
        results.extend(json_request["results"])

        while json_request["next"]:
            json_request = requests.get(json_request["next"]).json()
            results.extend(json_request["results"])
        return spark.createDataFrame(results)

    def custom_transform(self, df: DataFrame, table: str) -> DataFrame:
        return df.withColumn("properties", get_properties(F.col("url")))


bronze_instance = StarWarsBronze(spark, **options)

In [9]:
bronze_instance.load().transform().write(mode="overwrite").execute("people")

2025-03-16 07:36:32 | people | execute | Started
2025-03-16 07:36:32 | people | execute | Started
2025-03-16 07:36:32 | people | load | Started
2025-03-16 07:36:37 | people | load | Completed in 0.08 min
2025-03-16 07:36:37 | people | transform | Started
2025-03-16 07:36:38 | people | transform | Completed in 0.0 min
2025-03-16 07:36:38 | people | write | Started
2025-03-16 07:37:00 | people | write | Completed in 0.37 min
2025-03-16 07:37:00 | people | execute | Completed in 0.45 min
2025-03-16 07:37:00 | people | execute | Completed in 0.45 min


In [10]:
df = spark.sql(f"SELECT * FROM {CATALOG}.bronze.people")
print(f"No. Rows: {df.count()}")
df.show()

No. Rows: 82
+--------------------+-------------------+---+--------------------+--------------------+
|         LH_BronzeTS|               name|uid|                 url|          properties|
+--------------------+-------------------+---+--------------------+--------------------+
|2025-03-16 07:36:...|        Cliegg Lars| 62|https://www.swapi...|{"created": "2025...|
|2025-03-16 07:36:...|  Poggle the Lesser| 63|https://www.swapi...|{"created": "2025...|
|2025-03-16 07:36:...|    Luminara Unduli| 64|https://www.swapi...|{"created": "2025...|
|2025-03-16 07:36:...|      Barriss Offee| 65|https://www.swapi...|{"created": "2025...|
|2025-03-16 07:36:...|              Dormé| 66|https://www.swapi...|{"created": "2025...|
|2025-03-16 07:36:...|              Dooku| 67|https://www.swapi...|{"created": "2025...|
|2025-03-16 07:36:...|Bail Prestor Organa| 68|https://www.swapi...|{"created": "2025...|
|2025-03-16 07:36:...|         Jango Fett| 69|https://www.swapi...|{"created": "2025...|
|2025-03

# 2 Silver

In [11]:
ddl_schema_planets = "diameter STRING, rotation_period STRING, orbital_period STRING, gravity STRING, population STRING, climate STRING, terrain STRING, surface_water STRING, created STRING, edited STRING, name STRING, url STRING"
ddl_schema_people = "height STRING, mass STRING, hair_color STRING, skin_color STRING, eye_color STRING, birth_year STRING, gender STRING, created STRING, edited STRING, name STRING, homeworld STRING, url STRING"
DDL_SCHEMAS = {"planets": ddl_schema_planets, "people": ddl_schema_people}

In [12]:
options = {
    "catalog": CATALOG,
    "source_schema": "bronze",
    "target_schema": "silver",
}

In [13]:
class StarWarsSilver(silver.Silver):
    def custom_transform(self, df: DataFrame, table: str) -> DataFrame:
        df = df.withColumn(
            "properties", F.from_json(F.col("properties"), DDL_SCHEMAS[table])
        )
        df = df.withColumn("uid", F.col("uid").cast("int"))
        if table == "people":
            df = self.transf_people(df)
        return df

    def transf_people(self, df: DataFrame) -> DataFrame:
        df = (
            df.withColumn("height", df.properties.height)
            .withColumn("mass", df.properties.mass)
            .withColumn("gender", df.properties.gender)
            .drop("url", "properties")
        )
        for i in range(0, 35):
            df = df.withColumn(f"col{str(i)}", F.lit(str(i)))
        return df


silver_instance = StarWarsSilver(spark, **options)

In [14]:
silver_instance.load().transform().write(mode="overwrite", merge_schema=True).execute(
    "people"
)

2025-03-16 07:37:02 | people | execute | Started
2025-03-16 07:37:02 | people | execute | Started
2025-03-16 07:37:02 | people | load | Started
2025-03-16 07:37:02 | people | load | Completed in 0.0 min
2025-03-16 07:37:02 | people | transform | Started
2025-03-16 07:37:02 | people | transform | Completed in 0.0 min
2025-03-16 07:37:02 | people | write | Started
2025-03-16 07:37:04 | people | write | Completed in 0.03 min
2025-03-16 07:37:04 | people | execute | Completed in 0.03 min
2025-03-16 07:37:04 | people | execute | Completed in 0.03 min


In [15]:
df = spark.sql(f"SELECT * FROM {CATALOG}.silver.people")
print(f"No. Rows: {df.count()}")
df.show(100, truncate=False)

No. Rows: 82
+-------------------------+--------------------------+---------------------+---+-------+-------+-------------+----+----+----+----+----+----+----+----+----+----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+
|LH_SilverTS              |LH_BronzeTS               |name                 |uid|height |mass   |gender       |col0|col1|col2|col3|col4|col5|col6|col7|col8|col9|col10|col11|col12|col13|col14|col15|col16|col17|col18|col19|col20|col21|col22|col23|col24|col25|col26|col27|col28|col29|col30|col31|col32|col33|col34|
+-------------------------+--------------------------+---------------------+---+-------+-------+-------------+----+----+----+----+----+----+----+----+----+----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+
|2025-03-16 07:37:02.85472|2025-03-16 07:36:38.933913|

# 3 Optimize Silver

In [16]:
# Set layout for liquid columns and optimize undependend of write operation
(
    silver_instance.tblproperties(clusterby=["gender"])
    .optimize(optimize=True, vacuum=True)
    .execute("people")
)

2025-03-16 07:37:06 | people | execute | Started
2025-03-16 07:37:06 | people | execute | Started
2025-03-16 07:37:06 | people | load | Started
2025-03-16 07:37:06 | people | load | Completed in 0.0 min
2025-03-16 07:37:06 | people | transform | Started
2025-03-16 07:37:06 | people | transform | Completed in 0.0 min
2025-03-16 07:37:06 | people | write | Started
2025-03-16 07:37:08 | people | write | Completed in 0.02 min
2025-03-16 07:37:08 | people | tblproperties | Started
2025-03-16 07:37:23 | people | tblproperties | Completed in 0.25 min
2025-03-16 07:37:23 | people | optimize | Started
2025-03-16 07:37:46 | people | optimize | Completed in 0.37 min
2025-03-16 07:37:46 | people | execute | Completed in 0.65 min
2025-03-16 07:37:46 | people | execute | Completed in 0.65 min


In [17]:
# You can also do this with the default class
(
    silver.Silver(spark, **options)
    .tblproperties(clusterby=["gender"])
    .optimize(optimize=True, vacuum=True)
    .execute("people")
)

2025-03-16 07:37:46 | people | execute | Started
2025-03-16 07:37:46 | people | execute | Started
2025-03-16 07:37:46 | people | tblproperties | Started
2025-03-16 07:37:46 | people | tblproperties | Completed in 0.0 min
2025-03-16 07:37:46 | people | optimize | Started
2025-03-16 07:38:04 | people | optimize | Completed in 0.28 min
2025-03-16 07:38:04 | people | execute | Completed in 0.3 min
2025-03-16 07:38:04 | people | execute | Completed in 0.3 min


In [18]:
# run above commands also together with the write
(
    silver_instance.load()
    .transform()
    .write(mode="overwrite", merge_schema=True)
    .optimize(optimize=True, vacuum=True)
    .execute("people")
)

2025-03-16 07:38:04 | people | execute | Started
2025-03-16 07:38:04 | people | execute | Started
2025-03-16 07:38:04 | people | load | Started
2025-03-16 07:38:04 | people | load | Completed in 0.0 min
2025-03-16 07:38:04 | people | transform | Started
2025-03-16 07:38:04 | people | transform | Completed in 0.0 min
2025-03-16 07:38:04 | people | write | Started
2025-03-16 07:38:07 | people | write | Completed in 0.03 min
2025-03-16 07:38:07 | people | tblproperties | Started
2025-03-16 07:38:09 | people | tblproperties | Completed in 0.02 min
2025-03-16 07:38:09 | people | optimize | Started
2025-03-16 07:38:29 | people | optimize | Completed in 0.32 min
2025-03-16 07:38:29 | people | execute | Completed in 0.4 min
2025-03-16 07:38:29 | people | execute | Completed in 0.4 min


In [19]:
# run also a FULL Optimize when liquid clustering and a vacuum Lite with Delta 3.3 and above
(
    silver_instance.tblproperties(clusterby=["gender"])
    .optimize(optimize=True, optimize_full=True, vacuum=True, vacuum_lite=True)
    .execute("people")
)

2025-03-16 07:38:29 | people | execute | Started
2025-03-16 07:38:29 | people | execute | Started
2025-03-16 07:38:29 | people | load | Started
2025-03-16 07:38:29 | people | load | Completed in 0.0 min
2025-03-16 07:38:29 | people | transform | Started
2025-03-16 07:38:29 | people | transform | Completed in 0.0 min
2025-03-16 07:38:29 | people | write | Started
2025-03-16 07:38:32 | people | write | Completed in 0.05 min
2025-03-16 07:38:32 | people | tblproperties | Started
2025-03-16 07:38:34 | people | tblproperties | Completed in 0.02 min
2025-03-16 07:38:34 | people | optimize | Started
2025-03-16 07:38:44 | people | optimize | Completed in 0.15 min
2025-03-16 07:38:44 | people | execute | Completed in 0.23 min
2025-03-16 07:38:44 | people | execute | Completed in 0.23 min


In [20]:
# Set besides cluster cols, some integrated delta tblproperties and any other delta tblproperties as documented on https://docs.delta.io/
(
    silver_instance.tblproperties(
        clusterby=["gender"],
        deletion_vectors=True,
        auto_compact=True,
        optimize_write=True,
        change_data_feed=True,
        row_tracking=True,
        type_widening=True,
        tblproperties={"delta.enableChangeDataFeed": "true"},
    ).execute("people")
)

2025-03-16 07:38:44 | people | execute | Started
2025-03-16 07:38:44 | people | execute | Started
2025-03-16 07:38:44 | people | load | Started
2025-03-16 07:38:44 | people | load | Completed in 0.0 min
2025-03-16 07:38:44 | people | transform | Started
2025-03-16 07:38:44 | people | transform | Completed in 0.0 min
2025-03-16 07:38:44 | people | write | Started
2025-03-16 07:38:46 | people | write | Completed in 0.03 min
2025-03-16 07:38:46 | people | tblproperties | Started
2025-03-16 07:38:50 | people | tblproperties | Completed in 0.05 min
2025-03-16 07:38:50 | people | optimize | Started
2025-03-16 07:39:01 | people | optimize | Completed in 0.17 min
2025-03-16 07:39:01 | people | execute | Completed in 0.27 min
2025-03-16 07:39:01 | people | execute | Completed in 0.27 min


In [ ]:
# Run analyze and exclude cols you dont want an analyze to run. Especially long strings.
# Does not work with spark_catalog in Spark.
(
    silver_instance.tblproperties(clusterby=["gender"])
    .optimize(analyze=True, excl_cols=["url"])
    .execute("people")
)"


'\n(\n    silver_instance.tblproperties(clusterby=["gender"])\n    .optimize(analyze=True, excl_cols=["url"])\n    .execute("people")\n)"\n'

# Review in Delta History and details

In [22]:
from delta.tables import *

q = f"DESCRIBE HISTORY {CATALOG}.silver.people"
hist = spark.sql(q)
hist.show(truncate=False)

q = f"DESCRIBE DETAIL {CATALOG}.silver.people"
det = spark.sql(q)
det.show(truncate=False)

q = f"SHOW TBLPROPERTIES {CATALOG}.silver.people"
prop = spark.sql(q)
prop.show(truncate=False)

+-------+-----------------------+------+--------+---------------------------------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+----+--------+---------+-----------+-----------------+-------------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+------------+-----------------------------------+
|version|timestamp        

# 4 Clean Up

In [ ]:
spark.sql(f"DROP SCHEMA IF EXISTS {CATALOG}.bronze CASCADE")
spark.sql(f"DROP SCHEMA IF EXISTS {CATALOG}.silver CASCADE")
spark.sql(f"DROP SCHEMA IF EXISTS {CATALOG}.gold CASCADE")
spark.stop()

DataFrame[]